<h2><u>1. Equity Alpha Signal Model: Data Collection and Feature Engineering</u></h2>

**Author:** Farzana Khan Moutushi  
**Project:** Machine Learning Equity Alpha Signal Framework  
**Objective:** Predict future 20-day stock returns and develop alpha signals for portfolio backtesting.

<h2><u>1.1 Project Objective</u></h2>

In this notebook, I developed the first stage of an equity alpha signal framework. The primary objective was to collect historical market data, construct benchmark-relative return measures, engineer predictive financial features, and prepare a structured dataset for machine learning.

More broadly, this project was designed to answer whether machine learning models could identify stocks with stronger future 20-day return potential. Rather than treating the task as a simple price-prediction exercise, I framed it as an alpha research problem, where the central goal was to rank securities by expected future performance and later evaluate whether those rankings could support a systematic portfolio strategy.

## 1.2 Research Motivation

Equity alpha modeling requires more than observing historical stock prices. A stock may rise in absolute terms, but that does not necessarily imply that it generated meaningful alpha relative to the broader market. Therefore, I incorporated both individual stock behavior and benchmark-relative measures into the research design.

This approach allowed the project to move beyond descriptive price analysis and toward a more rigorous predictive framework. Specifically, I used historical stock data to engineer features related to momentum, volatility, technical positioning, and market-relative performance.

## 1.3 Notebook Scope

This notebook covers the data preparation stage of the project. The key tasks included:

1. Installing and importing the required Python libraries
2. Selecting the stock universe and benchmark
3. Downloading historical market data
4. Cleaning and saving the adjusted price dataset
5. Comparing total returns across securities
6. Preparing the foundation for feature engineering and machine learning modeling

The outputs from this notebook were later used for model development, backtesting, LSTM validation, and final signal selection.

## 1.4 Environment Setup

I began by setting up the Python environment and installing the required market data library. The `yfinance` package was used to retrieve historical stock and ETF price data directly into the notebook.

This step was necessary because the project relied on reproducible financial time-series data for return calculation, feature engineering, and downstream model training.

In [1]:
!pip install yfinance

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 19.3 MB/s  0:00:00

   ---------------------------------------- 0/5 [peewee]
   ---------------- ----------------------- 2/5 [websockets]
   ---------------- ----------------------- 2/5 [websockets]
   ------------------------ --------------- 3/5 [curl_cffi]
   ------------------------ --------------- 3/5 [curl_cffi]
   -------------------------------- ------- 4/5 [yfinance]
   -------------------------------- ------- 4/5 [yfinance]
   -------------------------------- ------- 4/5 [yfinance]
   ---------------------------------------- 5/5 [yfinance]



## 1.5 Importing Required Libraries

After installing the required package, I imported the core Python libraries used for data retrieval, data manipulation, numerical analysis, and visualization.

In [4]:
print(data.columns)

MultiIndex([( 'Close',  'AAPL'),
            ( 'Close',  'AMZN'),
            ( 'Close',   'BAC'),
            ( 'Close',   'BLK'),
            ( 'Close',     'C'),
            ( 'Close', 'GOOGL'),
            ( 'Close',    'GS'),
            ( 'Close',   'JPM'),
            ( 'Close',    'MS'),
            ( 'Close',  'MSFT'),
            ( 'Close',  'NVDA'),
            ( 'Close',   'QQQ'),
            ( 'Close',   'XOM'),
            (  'High',  'AAPL'),
            (  'High',  'AMZN'),
            (  'High',   'BAC'),
            (  'High',   'BLK'),
            (  'High',     'C'),
            (  'High', 'GOOGL'),
            (  'High',    'GS'),
            (  'High',   'JPM'),
            (  'High',    'MS'),
            (  'High',  'MSFT'),
            (  'High',  'NVDA'),
            (  'High',   'QQQ'),
            (  'High',   'XOM'),
            (   'Low',  'AAPL'),
            (   'Low',  'AMZN'),
            (   'Low',   'BAC'),
            (   'Low',   'BLK'),
          

In [5]:
prices = data["Close"]

In [6]:
import yfinance as yf
import pandas as pd

tickers = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "NVDA",
    "JPM", "GS", "MS", "BAC", "C", "BLK",
    "XOM",
    "QQQ"
]

data = yf.download(
    tickers,
    start="2019-01-01",
    end="2024-01-01"
)

# Use Close instead of Adj Close
prices = data["Close"]

print(prices.head())
print(prices.shape)

[*********************100%***********************]  13 of 13 completed


Ticker           AAPL       AMZN        BAC         BLK          C      GOOGL  \
Date                                                                            
2019-01-02  37.503735  76.956497  20.945084  325.040894  41.436672  52.301735   
2019-01-03  33.768082  75.014000  20.609425  315.492218  40.685802  50.853199   
2019-01-04  35.209602  78.769501  21.465357  327.044159  42.675190  53.461639   
2019-01-07  35.131245  81.475502  21.448570  327.953918  43.046749  53.355019   
2019-01-08  35.800953  82.829002  21.406614  332.127289  42.930641  53.823643   

Ticker              GS        JPM         MS       MSFT      NVDA         QQQ  \
Date                                                                            
2019-01-02  145.391663  80.836494  32.141228  94.397125  3.376983  148.040833   
2019-01-03  143.261902  79.687683  31.568422  90.924461  3.172956  143.204269   
2019-01-04  147.944046  82.625389  32.857258  95.153313  3.376239  149.331238   
2019-01-07  148.763840  82.

## 1.6 Stock Universe and Benchmark Design

I selected a focused universe of large-cap U.S. equities across technology, financials, communication services, consumer discretionary, and energy. This universe was intentionally compact so that the full alpha research workflow could remain interpretable while still allowing meaningful cross-sectional comparison across sectors.

I also included QQQ as the benchmark ETF because it tracks the Nasdaq-100 and provides a relevant market reference for many of the selected securities. In this project, QQQ was used to contextualize individual stock returns and later support market-relative feature engineering.

The ticker symbol identified each security in the dataset. For example, AAPL represented Apple, MSFT represented Microsoft, NVDA represented NVIDIA, and QQQ represented the benchmark ETF.

In [7]:
prices = prices.dropna()

In [8]:
prices.to_csv("prices_5y.csv")

In [9]:
print("\nData cleaned and saved successfully.")


Data cleaned and saved successfully.


In [10]:
end="2025-01-01"

## 1.7 Historical Market Data Collection

After defining the stock universe, I downloaded historical daily market data using `yfinance`. The objective of this step was to create a reproducible price dataset that could be used for return analysis, feature engineering, model training, and later backtesting.

I used adjusted or closing price data because return-based modeling requires consistent price observations across time. This step produced a price matrix where each row represented a trading date and each column represented a ticker.

In [11]:
import datetime

end_date = datetime.datetime.today()

data = yf.download(
    tickers,
    start="2019-01-01",
    end=end_date
)

[*********************100%***********************]  13 of 13 completed


In [12]:
end="2025-01-01"

In [13]:
prices = data["Close"]
prices = prices.dropna()

In [14]:
print(prices.tail())

Ticker            AAPL        AMZN        BAC          BLK           C  \
Date                                                                     
2026-04-29  270.170013  263.040009  52.880001  1039.380005  127.009201   
2026-04-30  271.350006  265.059998  53.459999  1065.599976  127.377457   
2026-05-01  280.140015  268.260010  53.240002  1061.680054  126.840004   
2026-05-04  276.829987  272.049988  52.189999  1052.250000  125.629997   
2026-05-05  280.280090  275.410004  53.244999  1064.459961  128.554993   

Ticker           GOOGL          GS         JPM          MS        MSFT  \
Date                                                                     
2026-04-29  349.940002  905.599976  309.250000  186.080002  424.459991   
2026-04-30  384.799988  923.770020  313.230011  190.589996  407.779999   
2026-05-01  385.690002  923.710022  312.470001  190.169998  414.440002   
2026-05-04  383.250000  903.270020  307.649994  188.009995  413.619995   
2026-05-05  388.489288  919.500000  3

## 1.8 Price Matrix Extraction and Data Export

After downloading the historical market data, I extracted the closing price matrix from the `Close` field. In the updated `yfinance` output, adjusted prices were already reflected through the download settings, so the `Adj Close` column was not separately available.

I then removed missing values to create an aligned dataset across all tickers. This step was necessary because return calculations, feature engineering, and later model training required each ticker to share the same trading-date structure.

Finally, I saved the cleaned price matrix as `prices_updated.csv` so that the next stages of the project could use a stable and reproducible input file.

In [15]:
# Extract closing prices again (updated data)
prices = data["Close"]

# Remove any missing values (clean dataset)
prices = prices.dropna()

# Save updated dataset to CSV file
prices.to_csv("prices_updated.csv")

print("Updated CSV saved successfully.")

Updated CSV saved successfully.


## 1.9 Total Return Comparison

After preparing the cleaned price matrix, I evaluated the total return of each security over the full sample period. This step provided an initial view of historical performance dispersion across the selected universe.

I used percentage return rather than nominal price level because a higher stock price does not necessarily indicate stronger investment performance. Total return measured how much each security appreciated relative to its own starting price.

The calculation followed the formula:

\[
\text{Total Return} = \frac{\text{Final Price}}{\text{Initial Price}} - 1
\]

This exploratory step was important because the later alpha model was designed to rank securities by expected future return, not by absolute price level.

In [17]:
# Calculate total return for each stock
# (Final price / Initial price) - 1
total_return = (prices.iloc[-1] / prices.iloc[0] - 1)

# Sort from highest return to lowest
total_return = total_return.sort_values(ascending=False)

# Print results
print(total_return)

Ticker
NVDA     57.529844
AAPL      6.473393
GOOGL     6.427848
GS        5.324297
MS        4.920743
QQQ       3.602580
MSFT      3.348965
JPM       2.843684
AMZN      2.578775
BLK       2.274849
C         2.102446
XOM       2.075287
BAC       1.542124
dtype: float64


## 1.10 Benchmark Return: QQQ

After comparing total returns across the full stock universe, I calculated the total return of QQQ separately. QQQ served as the market benchmark in this project because it represented broad Nasdaq-100 exposure and provided a reference point for evaluating stock-specific performance.

This benchmark return was important because alpha should not be interpreted as raw return alone. A stock may appreciate substantially, but the more relevant question is whether it performed better than the benchmark over the same period. Therefore, QQQ was later used to support benchmark-relative analysis and excess-return feature engineering.

In [18]:
# Get first and last price of QQQ
start_price = prices["QQQ"].iloc[0]
end_price = prices["QQQ"].iloc[-1]

# Calculate return
qqq_return = (end_price / start_price) - 1

print("QQQ return:", qqq_return)

QQQ return: 3.602580064104262


## 1.11 Notebook 1 Summary

In this notebook, I completed the foundational data preparation stage of the equity alpha signal project. I installed and imported the required libraries, defined a focused stock universe, downloaded historical market data, extracted the closing price matrix, removed missing values, and exported the cleaned dataset.

I also compared total returns across the selected securities and calculated QQQ’s return as the benchmark. This step established the empirical basis for later alpha analysis because the project was designed to evaluate not only raw stock performance, but also benchmark-relative return behavior.

The cleaned price dataset created in this notebook served as the input for the next stage of the project, where I engineered financial features, trained machine learning models, evaluated alpha rankings, and conducted backtesting.